In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import MetaTrader5 as mt5
import time
from datetime import datetime

In [11]:
"""
═══════════════════════════════════════════════════════════════════════════════
  EXTRAÇÃO DE DADOS DO WIN — MetaTrader 5
═══════════════════════════════════════════════════════════════════════════════

PRÉ-REQUISITOS:
  1. Terminal MT5 aberto e logado na conta demo
  2. pip install MetaTrader5 pandas

EXECUTE ESTE SCRIPT PRIMEIRO para:
  - Testar a conexão com o MT5
  - Descobrir o símbolo correto do WIN na sua corretora
  - Extrair o histórico de candles 1min
  - Salvar em CSV para usar no modelo

═══════════════════════════════════════════════════════════════════════════════
"""

import MetaTrader5 as mt5
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os

# ─────────────────────────────────────────────────────────────────────────────
# 1. CONEXÃO
# ─────────────────────────────────────────────────────────────────────────────

print("Conectando ao MetaTrader 5...")

if not mt5.initialize():
    print(f"ERRO: não foi possível conectar ao MT5.")
    print(f"Detalhes: {mt5.last_error()}")
    print("\nVerifique se o terminal MT5 está aberto e logado.")
    mt5.shutdown()
    exit()

info = mt5.terminal_info()
account = mt5.account_info()

print(f"\n✓ Conectado!")
print(f"  Terminal: {info.name}")
print(f"  Conta:    {account.login} ({account.server})")
print(f"  Tipo:     {'DEMO' if account.trade_mode == 0 else 'REAL'}")
print(f"  Saldo:    {account.balance:.2f} {account.currency}")


# ─────────────────────────────────────────────────────────────────────────────
# 2. DESCOBRIR O SÍMBOLO CORRETO DO WIN
# ─────────────────────────────────────────────────────────────────────────────
# Cada corretora usa um nome diferente para o contrato futuro do mini índice.
# Exemplos comuns: WIN$, WIN$N, WINM25, WING25, WINQ25, WIN@, MINI$
# Este bloco testa os candidatos e mostra quais existem na sua corretora.

print("\n─────────────────────────────────────────────────────")
print("Procurando símbolo do WIN na sua corretora...")

candidatos_win = [
    'WIN$', 'WIN$N', 'WIN@', 'MINI$',
    'WINM25', 'WINJ25', 'WINQ25', 'WINV25', 'WINZ25',
    'WING25', 'WINK25', 'WINM26',
]
candidatos_dol = [
    'WDO$', 'WDO$N', 'WDO@', 'WDOM25', 'WDOG25',
    'WDOJ25', 'WDOK25', 'WDOM26',
]

simbolos_disponiveis = [s.name for s in mt5.symbols_get()]

encontrados_win = [s for s in candidatos_win if s in simbolos_disponiveis]
encontrados_dol = [s for s in candidatos_dol if s in simbolos_disponiveis]

# busca mais ampla se não encontrou nenhum
if not encontrados_win:
    encontrados_win = [s for s in simbolos_disponiveis
                       if 'WIN' in s.upper() or 'MINI' in s.upper()]
if not encontrados_dol:
    encontrados_dol = [s for s in simbolos_disponiveis if 'WDO' in s.upper()]

print(f"\n  Símbolos WIN encontrados: {encontrados_win}")
print(f"  Símbolos WDO encontrados: {encontrados_dol}")

if not encontrados_win:
    print("\n  ATENÇÃO: Nenhum símbolo do WIN encontrado.")
    print("  Símbolos disponíveis que contêm 'WIN' ou 'IND':")
    for s in simbolos_disponiveis:
        if any(x in s.upper() for x in ['WIN', 'IND', 'MINI']):
            print(f"    {s}")
    mt5.shutdown()
    exit()

# usa o primeiro encontrado (contrato contínuo WIN$ é o ideal)
SIMBOLO_WIN = encontrados_win[0]
SIMBOLO_DOL = encontrados_dol[0] if encontrados_dol else None

print(f"\n  → Usando WIN: {SIMBOLO_WIN}")
print(f"  → Usando DOL: {SIMBOLO_DOL}")

# ativa os símbolos no mercado
mt5.symbol_select(SIMBOLO_WIN, True)
if SIMBOLO_DOL:
    mt5.symbol_select(SIMBOLO_DOL, True)


# ─────────────────────────────────────────────────────────────────────────────
# 3. EXTRAÇÃO DE CANDLES 1MIN
# ─────────────────────────────────────────────────────────────────────────────

print("\n─────────────────────────────────────────────────────")
print("Extraindo histórico de candles 1min...")

# quanto de histórico buscar
# GLPK consegue resolver bem com ~500-1000 candles de treino
# CPLEX aguenta 2000-5000 tranquilo
# Ajuste conforme a disponibilidade de dados da sua corretora demo

N_DIAS = 60  # comece com 30 dias; aumente se o solver aguentar

data_fim    = datetime.now()
data_inicio = data_fim - timedelta(days=N_DIAS)

def extrair_candles(simbolo, n_dias=60, timeframe=mt5.TIMEFRAME_M1):
    """Extrai candles OHLCV do MT5 e retorna DataFrame limpo."""
    data_fim    = datetime.now()
    data_inicio = data_fim - timedelta(days=n_dias)

    rates = mt5.copy_rates_range(simbolo, timeframe, data_inicio, data_fim)

    if rates is None or len(rates) == 0:
        print(f"  ERRO: sem dados para {simbolo}. {mt5.last_error()}")
        return None

    df = pd.DataFrame(rates)
    df['datetime'] = pd.to_datetime(df['time'], unit='s')
    df = df.rename(columns={'tick_volume': 'volume', 'real_volume': 'volume_fin'})
    df = df[['datetime', 'open', 'high', 'low', 'close', 'volume']]
    df = df.set_index('datetime')

    # filtra apenas horário de negociação do WIN: 9h00 às 17h55 (horário de Brasília)
    # o MT5 usa UTC, então ajusta: UTC-3 → 12h00-20h55 UTC
    df = df.between_time('12:00', '20:55')

    # remove candles com volume zero (leilão, circuit breaker, etc.)
    df = df[df['volume'] > 0]

    return df.reset_index()

# extrai WIN
df_win = extrair_candles(SIMBOLO_WIN, N_DIAS)
print(f"  WIN: {len(df_win)} candles extraídos")
print(f"  Período: {df_win['datetime'].min()} → {df_win['datetime'].max()}")
print(f"  Preço atual: {df_win['close'].iloc[-1]:.0f} pontos")
print(f"  Range médio: {(df_win['high'] - df_win['low']).mean():.0f} pontos")

# extrai DOL (opcional, para variável externa)
df_dol = None
if SIMBOLO_DOL:
    df_dol = extrair_candles(SIMBOLO_DOL, N_DIAS)
    if df_dol is not None:
        print(f"  DOL: {len(df_dol)} candles extraídos")


# ─────────────────────────────────────────────────────────────────────────────
# 4. ADICIONA VARIÁVEIS EXTERNAS E MINUTO DA SESSÃO
# ─────────────────────────────────────────────────────────────────────────────

print("\n─────────────────────────────────────────────────────")
print("Processando variáveis externas...")

df = df_win.copy()

# minuto da sessão: 0 = abertura (9h), 359 = último minuto (17h59)
# MT5 retorna em UTC, horário de Brasília = UTC-3
df['datetime_br'] = df['datetime'] - pd.Timedelta(hours=3)
df['minuto_sessao'] = (
    (df['datetime_br'].dt.hour - 9) * 60 + df['datetime_br'].dt.minute
).clip(0, 359)

# retorno do DOL (se disponível)
if df_dol is not None:
    df_dol['retorno_dol'] = df_dol['close'].pct_change() * 100
    # merge por datetime (inner join — só candles que existem nos dois)
    df = df.merge(
        df_dol[['datetime', 'retorno_dol']],
        on='datetime', how='left'
    )
    df['retorno_dol'] = df['retorno_dol'].fillna(0)
else:
    df['retorno_dol'] = 0.0
    print("  DOL não disponível — usando zero como placeholder")

# retorno do ES (S&P futuro) — não disponível em todas as contas demo B3
# se sua corretora tiver ES=F ou SP500 no MT5, use aqui
# caso contrário, fica zero e o modelo simplesmente não usa essa feature
df['retorno_es'] = 0.0

# tenta buscar S&P se disponível
sp_candidatos = ['ES$', 'ES=F', 'SP500', 'SPX500', 'US500', 'SP500m']
for sp in sp_candidatos:
    if sp in simbolos_disponiveis:
        df_sp = extrair_candles(sp, N_DIAS)
        if df_sp is not None:
            df_sp['retorno_es'] = df_sp['close'].pct_change() * 100
            df = df.merge(df_sp[['datetime','retorno_es']], on='datetime', how='left')
            df['retorno_es'] = df['retorno_es'].fillna(0)
            print(f"  S&P encontrado: {sp}")
            break
else:
    print("  S&P não encontrado — usando zero como placeholder")
    print("  (o modelo funciona, mas perde essa feature)")


# ─────────────────────────────────────────────────────────────────────────────
# 5. SALVA EM CSV
# ─────────────────────────────────────────────────────────────────────────────

output_path = 'win_1min_mt5.csv'
df.to_csv(output_path, index=False)

print(f"\n─────────────────────────────────────────────────────")
print(f"✓ Dados salvos em: {os.path.abspath(output_path)}")
print(f"  Linhas: {len(df)}")
print(f"  Colunas: {list(df.columns)}")
print(f"\nPrimeiras linhas:")
print(df[['datetime', 'open', 'high', 'low', 'close', 'volume',
          'minuto_sessao', 'retorno_dol', 'retorno_es']].head(5).to_string())
print(f"\nÚltimas linhas:")
print(df[['datetime', 'open', 'high', 'low', 'close', 'volume',
          'minuto_sessao', 'retorno_dol', 'retorno_es']].tail(5).to_string())

# ─────────────────────────────────────────────────────────────────────────────
# 6. DIAGNÓSTICO RÁPIDO DOS DADOS
# ─────────────────────────────────────────────────────────────────────────────

print(f"\n─────────────────────────────────────────────────────")
print("Diagnóstico dos dados:")
print(f"  Candles totais:         {len(df)}")
print(f"  Dias de pregão:         {df['datetime_br'].dt.date.nunique()}")
print(f"  Candles por dia (med):  {len(df) / df['datetime_br'].dt.date.nunique():.0f}")
print(f"  Range médio:            {(df['high']-df['low']).mean():.0f} pontos")
print(f"  Range máximo:           {(df['high']-df['low']).max():.0f} pontos")
print(f"  Volume médio:           {df['volume'].mean():.0f} contratos")

# candles por horário (padrão intradiário)
print(f"\n  Candles por faixa de horário:")
faixas = [(0,30,'Abertura (0-30min)'), (30,150,'Manhã (30-150min)'),
          (150,210,'Almoço (150-210min)'), (210,330,'Tarde (210-330min)'),
          (330,360,'Fechamento (330-360min)')]
for ini, fim, nome in faixas:
    mask = (df['minuto_sessao'] >= ini) & (df['minuto_sessao'] < fim)
    sub = df[mask]
    if len(sub) > 0:
        print(f"    {nome}: {len(sub)} candles | range médio: {(sub['high']-sub['low']).mean():.0f} pts")

mt5.shutdown()
print(f"\n✓ Conexão MT5 encerrada.")
print(f"\nPróximo passo: rode win_range_model.py apontando para '{output_path}'")

Conectando ao MetaTrader 5...

✓ Conectado!
  Terminal: MetaTrader 5
  Conta:    50493089 (XPMT5-DEMO)
  Tipo:     DEMO
  Saldo:    1000000.00 BRL

─────────────────────────────────────────────────────
Procurando símbolo do WIN na sua corretora...

  Símbolos WIN encontrados: ['WIN$', 'WIN$N', 'WIN@', 'WINM26']
  Símbolos WDO encontrados: ['WDO$', 'WDO$N', 'WDO@']

  → Usando WIN: WIN$
  → Usando DOL: WDO$

─────────────────────────────────────────────────────
Extraindo histórico de candles 1min...
  WIN: 15016 candles extraídos
  Período: 2026-04-09 12:00:00 → 2026-06-05 18:31:00
  Preço atual: 169520 pontos
  Range médio: 80 pontos
  DOL: 15210 candles extraídos

─────────────────────────────────────────────────────
Processando variáveis externas...
  S&P não encontrado — usando zero como placeholder
  (o modelo funciona, mas perde essa feature)

─────────────────────────────────────────────────────
✓ Dados salvos em: c:\Users\joaon\OneDrive\Documentos\GitHub\mestrado\Disciplina-mode

## APLICAÇÃO

In [12]:
"""
═══════════════════════════════════════════════════════════════════════════════
  MODELO DE PREVISÃO DE RANGE DO WIN — Dados Reais MT5
  Pyomo MIP | Cobertura ≥ 95% | GLPK / CPLEX
═══════════════════════════════════════════════════════════════════════════════

USO:
  1. Coloque este script na mesma pasta que win_1min_mt5.csv
  2. Ajuste CSV_PATH se necessário
  3. Execute: python win_range_model_mt5.py

SOLVER:
  - GLPK  (gratuito): apt-get install glpk-utils  /  conda install glpk
  - CPLEX (acadêmico): muito mais rápido para >1000 amostras
  Altere a variável SOLVER abaixo conforme o que tiver instalado.

═══════════════════════════════════════════════════════════════════════════════
"""

import numpy as np
import pandas as pd
import pyomo.environ as pyo
from pyomo.opt import SolverFactory
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURAÇÕES — ajuste aqui
# ─────────────────────────────────────────────────────────────────────────────

CSV_PATH       = 'win_1min_mt5.csv'   # caminho para o CSV gerado pelo mt5_win_dados.py
SOLVER         = 'cplex'               # 'glpk' ou 'cplex'
SOLVER_TIMEOUT = 300                  # segundos máximos para o solver

JANELA          = 15     # candles anteriores usados nas features de janela
COBERTURA_ALVO  = 0.95   # 95% dos candles reais devem ficar dentro do range previsto
DELTA_MIN       = 5      # range mínimo previsto em pontos (ajustado para WIN real ~80pts)
BIG_M           = 1500   # big-M (deve ser > maior range possível; WIN máx foi 640pts)
SPLIT_TREINO    = 0.80   # 80% treino, 20% teste

# features utilizadas
FEATURES = [
    # janela móvel — estado recente do mercado
    'atr',           # range médio dos últimos N candles
    'vol_real',      # desvio padrão dos retornos
    'range_medio',   # média do range (igual ATR aqui, redundante mas explícito)
    'momentum',      # close_t - close_{t-N}
    'vol_relativo',  # volume atual vs média — força do movimento
    # posição relativa
    'pos_janela',    # onde o preço está na janela (0=fundo, 1=topo)
    'pos_range',     # onde o close ficou dentro do candle (0=mínima, 1=máxima)
    'direcao',       # 1=alta, 0=baixa
    'gap',           # abertura vs fechamento anterior
    # padrão intradiário
    'hora_sin',      # componente senoidal do horário
    'hora_cos',      # componente cossenoidal
    'e_abertura',    # dummy: primeiros 30min
    'e_fechamento',  # dummy: últimos 30min
    'e_almoco',      # dummy: horário de almoço
    # variável externa
    'dol_retorno',   # retorno do WDO no candle (disponível nos seus dados)
    'dol_acum',      # retorno acumulado do WDO nos últimos N candles
]


# ─────────────────────────────────────────────────────────────────────────────
# 1. LEITURA E LIMPEZA DOS DADOS
# ─────────────────────────────────────────────────────────────────────────────

def carregar_dados(path):
    print(f"Carregando {path}...")
    df = pd.read_csv(path, parse_dates=['datetime'])

    # corrige minuto_sessao: recalcula a partir do horário real
    # o MT5 retorna em UTC; horário de Brasília = UTC-3
    if 'datetime_br' not in df.columns:
        df['datetime_br'] = df['datetime'] - pd.Timedelta(hours=3)

    df['datetime_br'] = pd.to_datetime(df['datetime_br'])

    # recalcula minuto da sessão corretamente
    # sessão do WIN: 9h00 às 17h55 (horário Brasília)
    hora_br  = df['datetime_br'].dt.hour
    min_br   = df['datetime_br'].dt.minute
    df['minuto_sessao'] = ((hora_br - 9) * 60 + min_br).clip(0, 359)

    # remove candles fora do pregão e volume zero
    df = df[(df['minuto_sessao'] >= 0) & (df['minuto_sessao'] <= 359)]
    df = df[df['volume'] > 0]

    # remove candles duplicados (pode acontecer na rolagem)
    df = df.drop_duplicates(subset='datetime')

    # ordena por tempo
    df = df.sort_values('datetime').reset_index(drop=True)

    # garante que retorno_dol existe
    if 'retorno_dol' not in df.columns:
        df['retorno_dol'] = 0.0
    df['retorno_dol'] = df['retorno_dol'].fillna(0.0)

    print(f"  {len(df)} candles | {df['datetime'].min().date()} → {df['datetime'].max().date()}")
    print(f"  Pregões:      {df['datetime_br'].dt.date.nunique()} dias")
    print(f"  Range médio:  {(df['high']-df['low']).mean():.1f} pontos")
    print(f"  Close atual:  {df['close'].iloc[-1]:.0f} pontos")

    return df


# ─────────────────────────────────────────────────────────────────────────────
# 2. ENGENHARIA DE FEATURES
# ─────────────────────────────────────────────────────────────────────────────

def construir_features(df_raw, janela=15):
    """
    Constrói todas as features deriváveis de candles OHLCV + WDO.
    Adiciona targets: mínima e máxima do PRÓXIMO candle.

    ATENÇÃO: o shift(-1) para os targets cruza a fronteira entre dias.
    Isso é corrigido zerando targets em candles de fim de dia.
    """
    d = df_raw.copy()

    # ── do candle atual ──────────────────────────────────────────────────────
    d['range']       = d['high'] - d['low']
    d['corpo']       = (d['close'] - d['open']).abs()
    d['direcao']     = (d['close'] >= d['open']).astype(int)
    d['pos_range']   = (d['close'] - d['low']) / d['range'].clip(lower=1)
    d['gap']         = d['open'] - d['close'].shift(1)
    d['retorno']     = d['close'].pct_change() * 100

    # ── janela móvel ─────────────────────────────────────────────────────────
    d['vol_real']    = d['retorno'].rolling(janela).std()
    d['atr']         = d['range'].rolling(janela).mean()
    d['range_medio'] = d['range'].rolling(janela).mean()
    d['momentum']    = d['close'] - d['close'].shift(janela)
    d['vol_relativo']= d['volume'] / d['volume'].rolling(janela).mean()
    d['max_janela']  = d['high'].rolling(janela).max()
    d['min_janela']  = d['low'].rolling(janela).min()
    d['pos_janela']  = (
        (d['close'] - d['min_janela']) /
        (d['max_janela'] - d['min_janela']).clip(lower=1)
    )

    # ── padrão intradiário ───────────────────────────────────────────────────
    d['hora_sin']     = np.sin(2 * np.pi * d['minuto_sessao'] / 360)
    d['hora_cos']     = np.cos(2 * np.pi * d['minuto_sessao'] / 360)
    d['e_abertura']   = (d['minuto_sessao'] < 30).astype(int)
    d['e_fechamento'] = (d['minuto_sessao'] > 330).astype(int)
    d['e_almoco']     = (
        (d['minuto_sessao'] > 150) & (d['minuto_sessao'] < 210)
    ).astype(int)

    # ── variáveis externas ───────────────────────────────────────────────────
    d['dol_retorno'] = d['retorno_dol']
    d['dol_acum']    = d['retorno_dol'].rolling(janela).sum()

    # ── targets: mínima e máxima do PRÓXIMO candle ───────────────────────────
    d['target_low']  = d['low'].shift(-1)
    d['target_high'] = d['high'].shift(-1)

    # CORREÇÃO IMPORTANTE: anula targets que cruzam a fronteira entre dias
    # (o próximo candle de sexta não é o primeiro candle de segunda)
    data_atual    = pd.to_datetime(d['datetime']).dt.date
    data_proxima  = pd.to_datetime(d['datetime']).shift(-1).dt.date
    cruzou_dia    = (data_atual != data_proxima)
    d.loc[cruzou_dia, 'target_low']  = np.nan
    d.loc[cruzou_dia, 'target_high'] = np.nan

    # remove NaN (janela inicial + fronteiras de dia + último candle)
    d = d.dropna(subset=FEATURES + ['target_low', 'target_high'])
    d = d.reset_index(drop=True)

    return d


# ─────────────────────────────────────────────────────────────────────────────
# 3. PRÉ-PROCESSAMENTO
# ─────────────────────────────────────────────────────────────────────────────

def preprocessar(df, features, split=0.80):
    """
    Normaliza features e converte targets para DESVIO relativo ao close.
    Trabalhar com desvio (em pontos) é mais estável numericamente e
    torna os coeficientes interpretáveis diretamente em pontos do WIN.

    Ex: beta0 = -45 significa que a mínima prevista fica, em média,
        45 pontos abaixo do close atual.
    """
    X_raw  = df[features].values.astype(float)
    close  = df['close'].values.astype(float)
    y_low  = df['target_low'].values.astype(float)
    y_high = df['target_high'].values.astype(float)

    # normalização z-score
    mu    = X_raw.mean(axis=0)
    sigma = X_raw.std(axis=0) + 1e-8
    X     = (X_raw - mu) / sigma

    # desvio em pontos relativo ao close atual
    dev_low  = y_low  - close   # negativo: mínima abaixo do close
    dev_high = y_high - close   # positivo: máxima acima do close

    n = int(len(X) * split)

    return {
        'X_tr':     X[:n],
        'X_te':     X[n:],
        'dL_tr':    dev_low[:n],
        'dL_te':    dev_low[n:],
        'dH_tr':    dev_high[:n],
        'dH_te':    dev_high[n:],
        'close_te': close[n:],
        'mu':       mu,
        'sigma':    sigma,
        'df_te':    df.iloc[n:].reset_index(drop=True),  # para análise posterior
    }


# ─────────────────────────────────────────────────────────────────────────────
# 4. MODELO PYOMO MIP
# ─────────────────────────────────────────────────────────────────────────────

def montar_e_resolver(dados, features,
                      cobertura_alvo=0.95, delta_min=5,
                      big_m=1500, solver='glpk', timeout=300):
    """
    Monta e resolve o MIP de previsão de range.

    FORMULAÇÃO:
    ─────────────────────────────────────────────────────────────────────────
    Variáveis:
      beta[k], beta0    coeficientes do modelo linear para a MÍNIMA
      gamma[k], gamma0  coeficientes do modelo linear para a MÁXIMA
      z[t] ∈ {0,1}      1 se o candle t foi coberto

    Objetivo:
      min  (1/T) Σ_t [ U_hat(t) - L_hat(t) ]
      → menor amplitude que ainda garante a cobertura desejada

    Restrições:
      (1/T) Σ_t z[t]  ≥  cobertura_alvo          cobertura mínima global
      L_hat(t) ≤ dL[t] + M·(1-z[t])              big-M mínima
      U_hat(t) ≥ dH[t] - M·(1-z[t])              big-M máxima
      U_hat(t) ≥ L_hat(t) + delta_min             restrição estrutural

    Onde:
      L_hat(t) = beta0  + Σ_k beta[k]·X[t,k]
      U_hat(t) = gamma0 + Σ_k gamma[k]·X[t,k]
    ─────────────────────────────────────────────────────────────────────────
    """
    X_tr  = dados['X_tr']
    dL_tr = dados['dL_tr']
    dH_tr = dados['dH_tr']
    T, K  = X_tr.shape

    print(f"\n{'─'*55}")
    print(f"  MODELO PYOMO MIP")
    print(f"{'─'*55}")
    print(f"  Amostras treino:   {T}")
    print(f"  Features:          {K}")
    print(f"  Cobertura alvo:    {cobertura_alvo:.0%}")
    print(f"  Delta mínimo:      {delta_min} pontos")
    print(f"  Big-M:             {big_m}")
    print(f"  Solver:            {solver}")
    print(f"  Timeout:           {timeout}s")

    model = pyo.ConcreteModel(name='WIN_Range_MIP')

    # variáveis de decisão
    model.beta   = pyo.Var(range(K), domain=pyo.Reals, initialize=0.0)
    model.gamma  = pyo.Var(range(K), domain=pyo.Reals, initialize=0.0)
    model.beta0  = pyo.Var(domain=pyo.Reals, initialize=float(dL_tr.mean()))
    model.gamma0 = pyo.Var(domain=pyo.Reals, initialize=float(dH_tr.mean()))
    model.z      = pyo.Var(range(T), domain=pyo.Binary)

    # expressões do modelo linear
    def L_hat(m, t):
        return m.beta0 + sum(m.beta[k] * float(X_tr[t, k]) for k in range(K))

    def U_hat(m, t):
        return m.gamma0 + sum(m.gamma[k] * float(X_tr[t, k]) for k in range(K))

    # função objetivo: minimizar amplitude média
    model.obj = pyo.Objective(
        expr=(1.0 / T) * sum(U_hat(model, t) - L_hat(model, t) for t in range(T)),
        sense=pyo.minimize
    )

    # restrição 1: cobertura mínima global
    model.cobertura = pyo.Constraint(
        expr=(1.0 / T) * sum(model.z[t] for t in range(T)) >= cobertura_alvo
    )

    # restrições 2 e 3: big-M
    model.cobre_low  = pyo.ConstraintList()
    model.cobre_high = pyo.ConstraintList()
    for t in range(T):
        model.cobre_low.add(
            L_hat(model, t) <= float(dL_tr[t]) + big_m * (1 - model.z[t])
        )
        model.cobre_high.add(
            U_hat(model, t) >= float(dH_tr[t]) - big_m * (1 - model.z[t])
        )

    # restrição 4: estrutural
    model.estrutura = pyo.ConstraintList()
    for t in range(T):
        model.estrutura.add(U_hat(model, t) >= L_hat(model, t) + delta_min)

    # resolve
    slv  = SolverFactory(solver)
    opts = {'tmlim': timeout} if solver == 'glpk' else {'timelimit': timeout}

    print(f"\n  Montando restrições e resolvendo...")
    result = slv.solve(model, tee=False, options=opts)
    status = str(result.solver.termination_condition)
    print(f"  Status: {status}")

    # extrai coeficientes
    coefs = {
        'beta0':  pyo.value(model.beta0),
        'gamma0': pyo.value(model.gamma0),
        'betas':  np.array([pyo.value(model.beta[k])  for k in range(K)]),
        'gammas': np.array([pyo.value(model.gamma[k]) for k in range(K)]),
    }

    cob_tr   = sum(pyo.value(model.z[t]) for t in range(T)) / T
    amp_tr   = pyo.value(model.obj)

    print(f"  Cobertura treino:  {cob_tr:.1%}")
    print(f"  Amplitude média:   {amp_tr:.1f} pontos")

    return coefs, status


# ─────────────────────────────────────────────────────────────────────────────
# 5. AVALIAÇÃO FORA DA AMOSTRA
# ─────────────────────────────────────────────────────────────────────────────

def avaliar(dados, coefs, cobertura_alvo=0.95):
    X_te     = dados['X_te']
    dL_te    = dados['dL_te']
    dH_te    = dados['dH_te']
    close_te = dados['close_te']
    df_te    = dados['df_te']

    L_pred = X_te @ coefs['betas']  + coefs['beta0']
    U_pred = X_te @ coefs['gammas'] + coefs['gamma0']

    low_abs  = close_te + L_pred
    high_abs = close_te + U_pred

    # cobertura: candle real ficou DENTRO do range previsto?
    coberto   = (low_abs <= close_te + dL_te) & (high_abs >= close_te + dH_te)
    cobertura = coberto.mean()
    amplitude = (high_abs - low_abs).mean()
    range_real= (dH_te - dL_te).mean()
    eficiencia= range_real / amplitude

    print(f"\n{'═'*55}")
    print(f"  AVALIAÇÃO FORA DA AMOSTRA")
    print(f"{'═'*55}")
    print(f"  Candles de teste:  {len(X_te)}")
    print(f"  Cobertura:         {cobertura:.1%}  (alvo: {cobertura_alvo:.0%})")
    print(f"  Amplitude prevista:{amplitude:.1f} pontos")
    print(f"  Range real médio:  {range_real:.1f} pontos")
    print(f"  Eficiência:        {eficiencia:.1%}  (range real / amplitude prevista)")
    print(f"  Violações:         {(~coberto).sum()} candles fora do range")

    # análise por faixa de horário
    print(f"\n  Cobertura por horário:")
    faixas = [
        (0,   30,  'Abertura  (0-30min)  '),
        (30,  150, 'Manhã     (30-150min)'),
        (150, 210, 'Almoço    (150-210min)'),
        (210, 330, 'Tarde     (210-330min)'),
        (330, 360, 'Fechamento(330-360min)'),
    ]
    ms = df_te['minuto_sessao'].values
    for ini, fim, nome in faixas:
        mask = (ms >= ini) & (ms < fim)
        if mask.sum() > 0:
            cob_h = coberto[mask].mean()
            amp_h = (high_abs - low_abs)[mask].mean()
            print(f"    {nome}: {cob_h:.1%} cobertura | {amp_h:.0f}pts amplitude")

    # últimos 10 candles do teste
    print(f"\n  Últimos 10 candles do teste (horário Brasília):")
    print(f"  {'Datetime':>18} | {'Low_real':>9} | {'High_real':>9} | "
          f"{'Low_prev':>9} | {'High_prev':>9} | {'ok?':>4}")
    print(f"  {'-'*68}")
    for i in range(max(-10, -len(X_te)), 0):
        dt  = str(df_te['datetime_br'].iloc[i])[:16] if 'datetime_br' in df_te.columns \
              else str(df_te['datetime'].iloc[i])[:16]
        lr  = round(close_te[i] + dL_te[i])
        hr  = round(close_te[i] + dH_te[i])
        lp  = round(low_abs[i])
        hp  = round(high_abs[i])
        ok  = "✓" if coberto[i] else "✗"
        print(f"  {dt:>18} | {lr:>9} | {hr:>9} | {lp:>9} | {hp:>9} | {ok:>4}")

    return cobertura, amplitude, range_real, coberto


# ─────────────────────────────────────────────────────────────────────────────
# 6. IMPORTÂNCIA DAS FEATURES
# ─────────────────────────────────────────────────────────────────────────────

def analisar_coeficientes(coefs, features):
    """
    Mostra os coeficientes do modelo de forma interpretável.
    Lembre-se: features estão normalizadas, então os coeficientes
    representam a sensibilidade do range previsto a 1 desvio padrão
    de cada feature — comparáveis entre si.
    """
    df_coef = pd.DataFrame({
        'feature':     features,
        'beta':        coefs['betas'],   # efeito sobre a mínima
        'gamma':       coefs['gammas'],  # efeito sobre a máxima
        'importancia': np.abs(coefs['betas']) + np.abs(coefs['gammas']),
    }).sort_values('importancia', ascending=False)

    print(f"\n{'─'*55}")
    print(f"  COEFICIENTES DO MODELO (normalizados)")
    print(f"  beta  = efeito sobre a MÍNIMA prevista (pontos)")
    print(f"  gamma = efeito sobre a MÁXIMA prevista (pontos)")
    print(f"{'─'*55}")
    print(f"  {'Feature':<18} | {'beta':>8} | {'gamma':>8} | {'|β|+|γ|':>8}")
    print(f"  {'-'*50}")
    for _, row in df_coef.iterrows():
        print(f"  {row['feature']:<18} | {row['beta']:>8.2f} | "
              f"{row['gamma']:>8.2f} | {row['importancia']:>8.2f}")

    print(f"\n  Interceptos:")
    print(f"  beta0  = {coefs['beta0']:>8.1f}  (desvio base da mínima em pontos)")
    print(f"  gamma0 = {coefs['gamma0']:>8.1f}  (desvio base da máxima em pontos)")

    return df_coef


# ─────────────────────────────────────────────────────────────────────────────
# 7. PREVISÃO ONLINE — candle a candle
# ─────────────────────────────────────────────────────────────────────────────

def prever_proximo_candle(features_dict, close_atual, coefs, mu, sigma):
    """
    Prevê o range [mínima, máxima] do próximo candle.

    Parâmetros:
        features_dict : dict com os valores das features do candle atual
        close_atual   : preço de fechamento do candle atual (pontos)
        coefs         : coeficientes retornados pelo modelo
        mu, sigma     : parâmetros de normalização do treino

    Retorna:
        (low_prev, high_prev) em pontos absolutos

    Uso prático (no loop de trading):
        # após fechar o candle atual:
        feats = calcular_features(janela_de_candles)
        low_p, high_p = prever_proximo_candle(feats, close, coefs, mu, sigma)
        print(f"Próximo candle: [{low_p}, {high_p}]")
    """
    x_raw = np.array([features_dict[f] for f in FEATURES], dtype=float)
    x     = (x_raw - mu) / sigma

    dev_low  = coefs['beta0']  + x @ coefs['betas']
    dev_high = coefs['gamma0'] + x @ coefs['gammas']

    return round(close_atual + dev_low), round(close_atual + dev_high)


# ─────────────────────────────────────────────────────────────────────────────
# EXECUÇÃO PRINCIPAL
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == '__main__':

    print("\n" + "═"*55)
    print("  WIN RANGE MODEL — Dados Reais MT5")
    print("═"*55)

    # 1. carrega dados
    print("\n[1/5] Carregando dados reais...")
    df_raw = carregar_dados(CSV_PATH)

    # 2. features
    print(f"\n[2/5] Construindo features (janela={JANELA})...")
    df = construir_features(df_raw, janela=JANELA)
    print(f"  {len(df)} candles válidos após dropna e correção de fronteiras")
    print(f"  {len(FEATURES)} features | targets: target_low, target_high")
    print(f"  Desvio médio da mínima: {(df['target_low']-df['close']).mean():.1f} pontos")
    print(f"  Desvio médio da máxima: {(df['target_high']-df['close']).mean():.1f} pontos")

    # 3. pré-processamento
    print(f"\n[3/5] Pré-processando (split={SPLIT_TREINO:.0%})...")
    dados = preprocessar(df, FEATURES, split=SPLIT_TREINO)
    print(f"  Treino: {len(dados['X_tr'])} candles")
    print(f"  Teste:  {len(dados['X_te'])} candles")

    # aviso de escala para o CPLEX
    T_treino = len(dados['X_tr'])
    if T_treino > 1500 and SOLVER == 'cplex':
        print(f"\n  ⚠ AVISO: {T_treino} amostras de treino com CPLEX pode ser lento.")
        print(f"  O solver vai rodar por até {SOLVER_TIMEOUT}s e retornar a melhor")
        print(f"  solução encontrada até lá (não necessariamente o ótimo global).")
        print(f"  Para escala maior, use CPLEX (acadêmico/gratuito para pesquisa).")
        print(f"  Alternativa: reduza N_TREINO_MAX abaixo para usar uma amostra.")

    # opcional: limita amostras de treino para o GLPK aguentar
    # descomente e ajuste se o solver travar:
    # N_TREINO_MAX = 800
    # if len(dados['X_tr']) > N_TREINO_MAX:
    #     print(f"  Limitando treino a {N_TREINO_MAX} amostras (mais recentes)...")
    #     for k in ['X_tr','dL_tr','dH_tr']:
    #         dados[k] = dados[k][-N_TREINO_MAX:]

    # 4. modelo
    print(f"\n[4/5] Resolvendo modelo MIP...")
    coefs, status = montar_e_resolver(
        dados, FEATURES,
        cobertura_alvo = COBERTURA_ALVO,
        delta_min      = DELTA_MIN,
        big_m          = BIG_M,
        solver         = SOLVER,
        timeout        = SOLVER_TIMEOUT,
    )

    # 5. avaliação
    print(f"\n[5/5] Avaliando fora da amostra...")
    cobertura, amplitude, range_real, coberto = avaliar(
        dados, coefs, cobertura_alvo=COBERTURA_ALVO
    )

    # coeficientes
    df_coef = analisar_coeficientes(coefs, FEATURES)

    # exemplo de previsão com o último candle disponível
    print(f"\n{'─'*55}")
    print(f"  PREVISÃO DO PRÓXIMO CANDLE")
    print(f"{'─'*55}")
    ultimo     = df.iloc[-1]
    feats_ult  = {f: ultimo[f] for f in FEATURES}
    close_ult  = ultimo['close']

    low_p, high_p = prever_proximo_candle(
        feats_ult, close_ult, coefs, dados['mu'], dados['sigma']
    )
    print(f"  Referência (close atual): {close_ult:.0f} pontos")
    print(f"  Mínima prevista:          {low_p} pontos  ({low_p - close_ult:+.0f})")
    print(f"  Máxima prevista:          {high_p} pontos  ({high_p - close_ult:+.0f})")
    print(f"  Amplitude prevista:       {high_p - low_p} pontos")
    print()

    # salva coeficientes para uso futuro
    import json
    coefs_salvar = {
        'beta0':  coefs['beta0'],
        'gamma0': coefs['gamma0'],
        'betas':  coefs['betas'].tolist(),
        'gammas': coefs['gammas'].tolist(),
        'mu':     dados['mu'].tolist(),
        'sigma':  dados['sigma'].tolist(),
        'features': FEATURES,
        'cobertura_treino': float(cobertura),
        'amplitude_media':  float(amplitude),
    }
    with open('win_range_coefs.json', 'w') as f:
        json.dump(coefs_salvar, f, indent=2)
    print(f"  Coeficientes salvos em: win_range_coefs.json")
    print(f"  (use para previsão online sem re-treinar)")


═══════════════════════════════════════════════════════
  WIN RANGE MODEL — Dados Reais MT5
═══════════════════════════════════════════════════════

[1/5] Carregando dados reais...
Carregando win_1min_mt5.csv...
  15016 candles | 2026-04-09 → 2026-06-05
  Pregões:      39 dias
  Range médio:  80.0 pontos
  Close atual:  169520 pontos

[2/5] Construindo features (janela=15)...
  14962 candles válidos após dropna e correção de fronteiras
  16 features | targets: target_low, target_high
  Desvio médio da mínima: -40.3 pontos
  Desvio médio da máxima: 39.5 pontos

[3/5] Pré-processando (split=80%)...
  Treino: 11969 candles
  Teste:  2993 candles

  ⚠ AVISO: 11969 amostras de treino com CPLEX pode ser lento.
  O solver vai rodar por até 300s e retornar a melhor
  solução encontrada até lá (não necessariamente o ótimo global).
  Para escala maior, use CPLEX (acadêmico/gratuito para pesquisa).
  Alternativa: reduza N_TREINO_MAX abaixo para usar uma amostra.

[4/5] Resolvendo modelo MIP...

